# Sprint 20 — Generate 4 Datasets for V9 Pipeline

**Mục tiêu:** Sinh đủ 4 datasets với seed cố định cho pipeline V9.

## 4-Dataset Strategy

| Dataset | Nội dung | Dùng cho |
|---------|----------|----------|
| `train_welding_robot.csv` | Normal only, reduced noise, deterministic | Training model |
| `val_train_welding_robot.csv` | Normal only, full noise (7 ngày) | Validate loss trong training loop |
| `threshold_calibration_welding_robot.csv` | Normal + 5 anomaly types, labelled | Chọn threshold thủ công |
| `test_welding_robot.csv` | Normal + 5 anomaly types, labelled | Evaluation cuối |

**Seed cố định:** `seed=42` cho mọi BatchGenerator.
**Output dir:** `data/generated_sprint20/`

In [ ]:
# %% Imports
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
# Walk up to find project root
for parent in [project_root] + list(project_root.parents):
    if (parent / "backend").is_dir():
        project_root = parent
        break
print(f"Project root: {project_root}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from backend.data_simulator.batch_generator import BatchGenerator
from backend.data_simulator.writer import CSVWriter
from datetime import datetime

print("All imports OK")

## Dataset 1: Training Data (Normal Only, Reduced Noise)

Dùng `BatchGenerator(training_mode=True)`:
- Normal only — không có anomaly nào
- Reduced noise + deterministic (cùng robot_id, program_id → cùng envelope)
- 60 ngày, 4 robots
- Dùng để training model

In [ ]:
# %% Generate training data (convergent mode, reduced noise)
import os

output_dir = str(project_root / "data" / "generated_sprint20")
os.makedirs(output_dir, exist_ok=True)
print(f"Output dir: {output_dir}")

train_gen = BatchGenerator(seed=42, training_mode=True)
writer = CSVWriter(output_dir=output_dir)

print("Generating training data (60 days, 4 robots, convergent, reduced-noise)...")
train_path, train_count = writer.write_train(
    train_gen,
    num_days=60,
    robots=[0, 1, 2, 3],
    start_date=datetime(2026, 1, 1),
)
print(f"Training data: {train_count:,} rows -> {train_path}")

## Dataset 2: Validation Training Data (Normal Only, Full Noise)

Dùng `BatchGenerator(seed=1023)` — **KHÔNG** `training_mode`:
- Normal only — không có anomaly
- **Full noise** — realistic sensor noise để validation loss phản ánh đúng điều kiện thực tế
- 7 ngày, 4 robots
- Dùng để validate loss trong training loop

In [ ]:
# %% Generate validation training data (normal only, full noise)
val_train_gen = BatchGenerator(seed=1023)  # NOT training_mode => full noise
val_train_path, val_train_count = writer.write_train(
    val_train_gen,
    num_days=7,
    robots=[0, 1, 2, 3],
    start_date=datetime(2026, 3, 1),
    output_filename="val_train_welding_robot.csv",
)
print(f"Validation training data: {val_train_count:,} rows -> {val_train_path}")

## Dataset 3: Threshold Calibration Data (Normal + Anomaly, Labelled)

Dùng `writer.write_threshold_calibration()`:
- 50% normal, 50% anomalous (5 types: spike, thermal_drift, flatline, cable_wear, arc_blow)
- Có labels: `threshold_calibration_labels.csv`
- Dùng để chọn threshold thủ công: plot distribution per sensor × per anomaly type
- Chọn threshold sao cho normal scores nằm dưới threshold và anomaly scores nằm trên

In [ ]:
# %% Generate threshold calibration data (normal + anomaly, labelled)
calib_gen = BatchGenerator(seed=42)
calib_csv_path, calib_label_path, calib_count = writer.write_threshold_calibration(
    calib_gen,
    num_cycles=2000,
    anomaly_ratio=0.5,
    robots=[0, 1, 2, 3],
    seed=42,
)
print(f"Threshold calibration data: {calib_count:,} rows -> {calib_csv_path}")
print(f"Calibration labels: -> {calib_label_path}")

## Dataset 4: Test Data (Normal + Anomaly, Labelled)

Dùng `writer.write_test()`:
- 50% normal, 50% anomalous (5 types)
- Có labels: `test_labels.csv`
- Dùng cho evaluation cuối: tính AUROC, precision, recall, F1

In [ ]:
# %% Generate test data (normal + anomaly, labelled)
test_gen = BatchGenerator(seed=42)
test_csv_path, test_label_path, test_count = writer.write_test(
    test_gen,
    num_cycles=8000,
    anomaly_ratio=0.5,
    robots=[0, 1, 2, 3],
    seed=42,
    start_date=datetime(2026, 5, 1),
)
print(f"Test data: {test_count:,} rows -> {test_csv_path}")
print(f"Test labels: -> {test_label_path}")

## Verification: Label Distribution & Step Counts

Kiểm tra:
1. Mỗi dataset có đúng số cycles và steps
2. Anomaly types phân bố đều trong threshold_calibration và test
3. Mọi cycle đều có 512 steps
4. Chỉ 5 anomaly types + 'normal' xuất hiện (không có archived types)

In [ ]:
# %% Verify label distribution and step counts
import pandas as pd
import numpy as np

# Load labels for calibration and test sets
calib_labels = pd.read_csv(calib_label_path)
test_labels = pd.read_csv(test_label_path)

print("=== THRESHOLD CALIBRATION LABEL DISTRIBUTION ===")
print(calib_labels["anomaly_type"].value_counts())
print(f"Normal: {(calib_labels['is_anomaly'] == 0).sum()}")
print(f"Anomaly: {(calib_labels['is_anomaly'] == 1).sum()}")
print()

print("=== TEST LABEL DISTRIBUTION ===")
print(test_labels["anomaly_type"].value_counts())
print(f"Normal: {(test_labels['is_anomaly'] == 0).sum()}")
print(f"Anomaly: {(test_labels['is_anomaly'] == 1).sum()}")
print()

# Verify step counts
for name, path in [
    ("Train", train_path),
    ("Val Train", val_train_path),
    ("Threshold Calib", calib_csv_path),
    ("Test", test_csv_path),
]:
    df = pd.read_csv(path)
    steps = df.groupby("cycle_id").size()
    min_s, max_s, uniq = steps.min(), steps.max(), steps.nunique()
    status = "OK" if (min_s == 512 and max_s == 512 and uniq == 1) else "FAIL"
    print(f"{name:20s}: {len(df):>8,} rows, {steps.nunique()} unique step counts, "
          f"min={min_s}, max={max_s} [{status}]")

# Verify anomaly types (only 5 + normal)
test_anom_types = set(test_labels[test_labels["is_anomaly"] == 1]["anomaly_type"].unique())
calib_anom_types = set(calib_labels[calib_labels["is_anomaly"] == 1]["anomaly_type"].unique())
print(f"\nTest anomaly types: {sorted(test_anom_types)} ({len(test_anom_types)})")
print(f"Calib anomaly types: {sorted(calib_anom_types)} ({len(calib_anom_types)})")

archived = {"phase_shift", "wire_stutter", "gas_blockage", "sticky_tc", "overshoot_degradation"}
for name, types in [("Test", test_anom_types), ("Calib", calib_anom_types)]:
    found = types & archived
    if found:
        print(f"WARNING [{name}]: Archived types found: {found}")
    else:
        print(f"OK [{name}]: No archived types")

print("\n=== ALL VERIFICATIONS PASSED ===")